# Traductor de txt a SQL

**Índice**   
1. [Imports](#imports)
2. [Cargamos los modelos](#cargamos-los-modelos)
3. [Creamos la base de datos](#creamos-la-base-de-datos)
4. [Mapeo de columnas](#mapeo-de-columnas)
5. [Sinonimos/palabras clave (ES/EN)](#sinonimos--palabras-clave-esen)
6. [Metricas](#metricas)
7. [Agrupaciones temporales](#agrupaciones-temporales)
8. [Funcion que detecta el idioma](#funcion-que-detecta-el-idioma) .
9. [Filtro de fecha](#filtro-de-fecha)
10. [Filtro minus y mayus](#filtro-minus-y-mayus)
11. [Filtro agrupaciones](#filtro-agrupaciones)
12. [Detección de where](#detección-de-where).
13. [Detección de group](#detección-de-group).
14. [Detección de filtro](#detección-de-filtro).
15. [Generador de SQL](#generador-de-SQL)
16. [Pruebas](#pruebas)

## Imports

In [752]:
import re
import spacy
from langdetect import detect
import pandas as pd

## Cargamos los modelos 

In [753]:
# Modelos
nlp_es = spacy.load("es_core_news_sm") # Español
nlp_en = spacy.load("en_core_web_sm") # English

## Creamos la base de datos

In [754]:
# Cargar los CSVs (ajusta las rutas a tus archivos locales)
clientes = pd.read_csv("../data/clientes_ecommerce.csv")
transacciones = pd.read_csv("../data/transacciones_ecommerce.csv")

In [755]:
df = pd.merge(transacciones, clientes, on="id_cliente", how="outer")
TABLE_NAME = "merge_transaccion_cliente"
# IMPORTANTE: en tu df mergeado las columnas son las del CSV, aquí asumo que usas las españolas.

In [756]:
df

,id_transaccion,id_cliente,fecha_compra,producto,categoria_producto,precio_unitario,cantidad,importe_total,metodo_pago,coste_envio,coste_fabricacion,nombre,apellidos,email,pais,ciudad,edad,genero
0,1308.0,1,2024-03-21,Xiaomi 13,Móviles,1269.37,2.0,2538.74,bizum,6.64,723.00,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
1,1423.0,1,2023-09-04,Apple Watch Series 9,Relojes inteligentes,561.03,1.0,561.03,paypal,8.13,301.15,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
2,7682.0,1,2023-08-16,Auriculares Sony WH-1000XM5,Accesorios,224.99,1.0,224.99,tarjeta,5.90,67.77,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
3,8220.0,1,2024-10-26,Lenovo ThinkPad X1 Carbon,Portátiles,1562.94,2.0,3125.88,transferencia,24.97,1166.97,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
4,9001.0,1,2023-02-26,Huawei Watch GT 4,Relojes inteligentes,264.37,3.0,793.11,bizum,7.82,141.60,Carlos,Sánchez Ramos,carlos.sanchez@gmail.com,España,Barcelona,29,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15229,386.0,4999,2024-10-05,Apple Watch Series 9,Relojes inteligentes,437.23,2.0,874.46,transferencia,8.11,204.23,Alberto,Jiménez Ruiz,alberto.jimenez2@live.com,Reino Unido,Londres,49,F
15230,1958.0,4999,2024-09-04,Auriculares Sony WH-1000XM5,Accesorios,10.27,2.0,20.54,paypal,4.76,3.91,Alberto,Jiménez Ruiz,alberto.jimenez2@live.com,Reino Unido,Londres,49,F
15231,2311.0,4999,2024-02-21,Xiaomi 13,Móviles,739.49,2.0,1478.98,bizum,12.94,464.58,Alberto,Jiménez Ruiz,alberto.jimenez2@live.com,Reino Unido,Londres,49,F
15232,8452.0,4999,2023-09-20,HP Spectre x360,Portátiles,2215.25,1.0,2215.25,tarjeta,16.57,1340.05,Alberto,Jiménez Ruiz,alberto.jimenez2@live.com,Reino Unido,Londres,49,F


## Mapeo de columnas

In [757]:
COLS = {
    "id_cliente","nombre","apellidos","email","pais","ciudad","edad","genero",
    "id_transaccion","fecha_compra","producto","categoria_producto",
    "precio_unitario","cantidad","importe_total","metodo_pago",
    "coste_envio","coste_fabricacion"
}

## Sinonimos / palabras clave (ES/EN)

In [758]:
SYN_TO_COL = {
    "es": {
        # importe total Ventas / Dinero
        "ventas": "importe_total",
        "dinero": "importe_total",
        "ingresos": "importe_total",
        "beneficios": "importe_total",
        "ganancias": "importe_total",   
        "facturacion": "importe_total",
        "importe": "importe_total",
        "total": "importe_total",
        "monto": "importe_total",      
        "recaudacion": "importe_total", 
        "caja": "importe_total",        
        "pasta": "importe_total",       
        "plata": "importe_total",      
        "valor": "importe_total",
        #cantidad / unidades
        "unidades": "cantidad",
        "cantidad": "cantidad",
        "numero": "cantidad",           
        "volumen": "cantidad",          
        "cuantos": "cantidad",          
        "cuantas": "cantidad",
        #precio
        "precio": "precio_unitario",
        "coste": "precio_unitario",     
        "valor_unitario": "precio_unitario",
        "pvp": "precio_unitario",
      # Costes específicos
        "envio": "coste_envio",
        "transporte": "coste_envio",
        "portes": "coste_envio",
        "logistica": "coste_envio",
        "entregas": "coste_envio",
        "fabricacion": "coste_fabricacion",
        "produccion": "coste_fabricacion",
        "elaboracion": "coste_fabricacion",
        # Ubicación (pais / ciudad)
        "pais": "pais",
        "nacion": "pais",
        "region": "pais",    
        "territorio": "pais",
        "ciudad": "ciudad",
        "capital": "ciudad",
        "municipio": "ciudad",
        "ubicacion": "ciudad",   
        "localidad": "ciudad",
        # Producto y Categoría
        "producto": "producto",
        "articulo": "producto",
        "item": "producto",
        "modelo": "producto",
        "referencia": "producto",
        "categoria": "categoria_producto",
        "tipo": "categoria_producto",
        "clase": "categoria_producto",
        "familia": "categoria_producto", 
        "seccion": "categoria_producto", 
        "gama": "categoria_producto",
        # Cliente (Personas)
        "cliente": "id_cliente",
        "comprador": "id_cliente",
        "usuario": "id_cliente",
        "consumidor": "id_cliente",
        "persona": "id_cliente",
        "clientes": "id_cliente",
        # Demografía
        "genero": "genero",
        "sexo": "genero",
        "hombres": "genero",           
        "mujeres": "genero",
        "edad": "edad",
        "anos": "edad",               
        "nacimiento": "edad",          
        "viejo": "edad",              
        "joven": "edad",
        # Transacción y Fechas
        "transaccion": "id_transaccion",
        "pedido": "id_transaccion",
        "orden": "id_transaccion",
        "ticket": "id_transaccion",
        "factura": "id_transaccion",
        "operacion": "id_transaccion",
        "venta": "id_transaccion",
        "metodo": "metodo_pago",
        "pago": "metodo_pago",
        "forma": "metodo_pago",        
        "tarjeta": "metodo_pago",     
        "efectivo": "metodo_pago",
        "fecha": "fecha_compra",
        "compra": "fecha_compra",
        "dia": "fecha_compra",
        "cuando": "fecha_compra",      
        "momento": "fecha_compra"
    },
    "en": {
        # si el usuario pregunta en inglés, seguimos generando SQL con columnas ES
        # (porque tu dataset está en ES). Solo traducimos la intención.
   # Sales / Money importe_total
        "sales": "importe_total",
        "revenue": "importe_total",
        "income": "importe_total",
        "earnings": "importe_total",
        "profit": "importe_total",     
        "turnover": "importe_total",  
        "amount": "importe_total",
        "total": "importe_total",
        "money": "importe_total",
        "value": "importe_total",       
        "billings": "importe_total",
        # Quantity -> cantidad
        "units": "cantidad",
        "quantity": "cantidad",
        "volume": "cantidad",           
        "count": "cantidad",
        "number": "cantidad",           
        "items": "cantidad",            
        # Price -> precio_unitario
        "price": "precio_unitario",
        "cost": "precio_unitario",      
        "unit": "precio_unitario",      
        "rate": "precio_unitario",
        "worth": "precio_unitario",     
        # Costs
        "shipping": "coste_envio",
        "delivery": "coste_envio",
        "transport": "coste_envio",
        "freight": "coste_envio",
        "logistics": "coste_envio",
        "manufacturing": "coste_fabricacion",
        "production": "coste_fabricacion",
        "making": "coste_fabricacion",  
        # Location -> pais / ciudad
        "country": "pais",
        "nation": "pais",
        "region": "pais",
        "territory": "pais",
        "land": "pais",
        
        "city": "ciudad",
        "town": "ciudad",
        "location": "ciudad",
        "municipality": "ciudad",
        "village": "ciudad",

        # Product
        "product": "producto",
        "item": "producto",
        "article": "producto",
        "model": "producto",
        "sku": "producto",
        "good": "producto",           
        
        "category": "categoria_producto",
        "type": "categoria_producto",
        "class": "categoria_producto",
        "family": "categoria_producto",
        "kind": "categoria_producto",   
        "group": "categoria_producto",

        # Demographics
        "gender": "genero",
        "sex": "genero",
        "male": "genero",
        "female": "genero",
        
        "age": "edad",
        "years": "edad",                
        "old": "edad",                  

        # Transaction details
        "payment": "metodo_pago",
        "method": "metodo_pago",
        "card": "metodo_pago",         
        "cash": "metodo_pago",
        
        "date": "fecha_compra",
        "purchase": "fecha_compra",
        "time": "fecha_compra",
        "when": "fecha_compra",
        "day": "fecha_compra",

        # IDs
        "transaction": "id_transaccion",
        "order": "id_transaccion",
        "deal": "id_transaccion",
        "invoice": "id_transaccion",
        "ticket": "id_transaccion",
        
        "client": "id_cliente",
        "customer": "id_cliente",
        "user": "id_cliente",
        "buyer": "id_cliente",
        "shopper": "id_cliente"
    }
}

## Meses

In [759]:
MONTHS = {
    "es": {
        "enero": 1, "febrero": 2, "marzo": 3, "abril": 4,
        "mayo": 5, "junio": 6, "julio": 7, "agosto": 8,
        "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12,
    },
    "en": {
        "january": 1, "february": 2, "march": 3, "april": 4,
        "may": 5, "june": 6, "july": 7, "august": 8,
        "september": 9, "october": 10, "november": 11, "december": 12,
    }
}

## Mapeo de paises

In [996]:
PAIS_MAP = {
    "mexico": "México",
    "argentina": "Argentina",
    "espana": "España",
    "españa": "España",
    "en reino unido": "Reino Unido",
    "francia": "Francia",
    "portugal": "Portugal",
    "alemania": "Alemania",
    "italia": "Italia",

    # Inglés
    "mexico": "México",
    "argentina": "Argentina",
    "spain": "España",
    "united kingdom": "Reino Unido",
    "uk": "Reino Unido",
    "england": "Reino Unido",
    "france": "Francia",
    "portugal": "Portugal",
    "germany": "Alemania",
    "italy": "Italia"

}

## Mapeo de categoria

In [761]:
CATEGORIA_MAP = {
    "accesorios": "Accesorios",
    "accesorio": "Accesorios",

    "reloj": "Relojes inteligentes",
    "relojes": "Relojes inteligentes",
    "reloj inteligente": "Relojes inteligentes",
    "smartwatch": "Relojes inteligentes",

    "movil": "Móviles",
    "moviles": "Móviles",
    "telefono": "Móviles",
    "telefonos": "Móviles",
    "smartphone": "Móviles",

    "portatil": "Portátiles",
    "portatiles": "Portátiles",
    "pc": "Portátiles",
    "laptop": "Portátiles",
    "ordenador": "Portátiles",
    "computadora": "Portátiles",
}

## Metricas

FALTAN ¿Varianza?

In [1149]:
AGG_WORDS = {
    "es": {
        "avg": {"promedio", "media", "promediar","valor medio" },
        "sum": {"suma", "total", "sumar", "sumatorio", "acumulado", "agregado",},
        "count": {"cuantos", "cuantas", "numero","veces", "numeros", "conteo", "contar"},
        "max": {"maximo", "maxima", "pico", "tope", "mejor"}, # Sin max, para evitar falsos max
        "min": {"minimo", "minima", "bajo", "peor"}, # Sin min, para evitar falsos min
        "median": {"mediana", "percentil", "percentil 50","valor central"},
        "mode": {"moda", "mas frecuente", "frecuente","habitual", "tendencia"},
        "std": {"desviacion", "desviacion estandar", "variacion", "volatilidad", "dispersion",},
        "var": {"varianza", "variance", "var", "variacion", "variabilidad"},

    },
    "en": {
        "avg": {"average", "avg", "mean"},
        "sum": {"sum", "total"},
        "count": {"count", "how", "many", "number"},
        "max": {"max", "maximum", "highest", "top"},
        "min": {"min", "minimum", "lowest"},
        "median": {"median", "percentile"},
        "mode": {"mode", "most frequent"},
        "std": {"std", "stddev", "standard deviation"},
        "var": {"variance", "var", "variability"},

    }
}

## Agrupaciones temporales

In [763]:
TIME_GROUP_WORDS = {
    "es": {
        "quarter": {"trimestre", "trimestral", "trimestralmente"},
        "month": {"mes", "mensual"},
        "year": {"año", "ano", "anual"},
    },
    "en": {
        "quarter": {"quarter", "qtr"},
        "month": {"month", "monthly"},
        "year": {"year", "yearly", "annual"},
    }
}

## Mapeo para rankings

In [ ]:
RANKING_WORDS = {
"es": {
        # 1. RANKING POR VOLUMEN (Gente que quiere ver movimiento de stock) Se mapea a: COUNT o SUM(cantidad)
        "cantidad": {
            # Basicos
            "mas vendido", "más vendido", "mas vendidos", "más vendidos",
            "top vendidos", "top ventas",
            "productos mas vendidos",
            
            # Negocio / Inventario
            "mayor volumen", "mayor volumen de ventas",
            "mayor rotacion", "mayor rotación", 
            "mas populares", "más populares",   
            "mas demandados", "más demandados",
            "numero uno", "número uno",
            "preferidos", "favoritos"
        },

        # 2. RANKING MONETARIO (Gente que quiere ver dinero)Se mapea a: SUM(importe_total)
        "importe_total": {
            # Básicos
            "mayores ventas", "mayor venta",
            "mas ingresos", "más ingresos",
            "mayores ingresos",
            "facturacion mas alta", "facturación más alta",
            "mayor facturacion", "mayor facturación",
            
            # Negocio / Financiero
            "mas rentables", "más rentables",  
            "mejor rendimiento",
            "mas valiosos", "más valiosos",
            "recaudacion mas alta", "recaudación más alta",
            "mayor impacto",
            "top ingresos",
            "dinero generado"
        },

        # 3. RANKING POR VALOR DEL PRODUCTO (Nuevo: ¿Cuál es el más caro?)mapea a: MAX(precio_unitario) u ORDER BY precio_unitario
        "precio_unitario": {
            "mas caros", "más caros",
            "mas caro", "más caro",
            "mayor precio", "precio mas alto",
            "mas costosos", "más costosos",
            "gama alta", "premium",
            "mayor valor unitario"
        }
    },
    "en": {
# 1. VOLUME RANKING (Quantity)
        "cantidad": {
            # Basic
            "best selling", "best seller",
            "most sold",
            "top selling", "top sellers",
            
            # Business / Stock
            "highest volume", "highest sales volume",
            "most popular",              
            "in high demand",
            "most frequent",
            "number one",
            "market leader"
        },

        # 2. MONETARY RANKING (Revenue)
        "importe_total": {
            # Basic
            "highest revenue", "top revenue",
            "most revenue",
            "top sales", "highest sales",
            
            # Business / Financial
            "highest earning", "top earning",
            "most profitable",          
            "highest grossing",           
            "best performing",
            "top financial",
            "money makers"
        },

        # 3. PRICE RANKING (Unit Price)
        "precio_unitario": {
            "most expensive",
            "highest price", "highest priced",
            "costliest",
            "premium",
            "high end",
            "top tier"
        }
    }
}

## Funcion que detecta el idioma

In [1010]:
def detectar_idioma(texto: str):
    lang = detect(texto)
    if lang == "es":
        return nlp_es(texto), "es"
    elif lang == "en":
        return nlp_en(texto), "en"
    else:
        return nlp_es(texto), "es"

## Prepocesamiento del texto

### Quitar tíldes

In [766]:
import unicodedata

def strip_accents(text: str) -> str:
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )

### Prepocesado minus y mayus

In [767]:
def _normalize_tokens(doc):
    # lemmas en minúscula, sin puntuación/espacios
        return [
        strip_accents(t.lemma_.lower())
        for t in doc
        if not t.is_punct and not t.is_space
    ]

## Filtro de fecha

### Filtro año

In [768]:
def _find_years(texto: str):
    return sorted(set(re.findall(r"\b(2023|2024)\b", texto)))

### SQL año

In [769]:
def _year_range_condition_pg(years):
    # years es lista de strings [“2023”] o [“2023",“2024"]
    start_y = min(years)
    end_y = max(years)
    return (
        f"fecha_compra BETWEEN '{start_y}-01-01' AND '{end_y}-12-31'"
    )

### Rango meses

In [770]:
def _find_month_ranges(texto: str, idioma: str):
    texto = texto.lower()
    pattern = (
        r"(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre|"
        r"january|february|march|april|may|june|july|august|september|october|november|december)"
        r"\s+(a|y|hasta|to|and)\s+"
        r"(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre|"
        r"january|february|march|april|may|june|july|august|september|october|november|december)"
        r"(?:\s+(\d{4}))?"
    )
    matches = re.findall(pattern, texto)
    # Devuelve mes_inicio, mes_fin, año (como int o None)
    result = []
    for m_start, _, m_end, year in matches:
        y = int(year) if year else None
        result.append((m_start, m_end, y))
    return result

### SQL rango meses

In [771]:
from calendar import monthrange

def _month_range_condition_pg(start_month, end_month, year):
    start_date = f"{year}-{start_month:02d}-01"
    # último día del mes final
    last_day = monthrange(int(year), end_month)[1]
    end_date = f"{year}-{end_month:02d}-{last_day}"
    return f"fecha_compra BETWEEN '{start_date}' AND '{end_date}'"


### Filtro un mes en concreto de un año en concreto

In [772]:
def _find_single_month(texto: str, idioma: str):
    texto = texto.lower()
    for m, num in MONTHS[idioma].items():
        m_match = re.search(rf"\b{m}\b(?:\s+(?:de|del|of))?\s*(\d{{4}})?", texto)
        if m_match:
            year = m_match.group(1)
            return num, year
    return None

### SQL un mes en concreto de un año en concreto

In [773]:
def _single_month_condition_pg(month, year):
    start_date = f"{year}-{month:02d}-01"
    end_day = monthrange(int(year), month)[1]
    end_date = f"{year}-{month:02d}-{end_day}"
    return f"fecha_compra BETWEEN '{start_date}' AND '{end_date}'"

### Fechas relativas

In [1049]:
from calendar import monthrange
import re

def _last_day_of_month(year, month):
    return monthrange(year, month)[1]

def _relative_time_condition(doc, idioma: str, available_years=[2023, 2024]):
    texto = strip_accents(doc.text.lower())

    years_in_text = _find_years(texto)
    year = int(years_in_text[0]) if years_in_text else None

    # TRIMESTRE
    m_quarter = re.search(r"(ultim(?:o|a|os|as)?|primer(?:o|a|os|as)?|last|first)\s+(trimestre|quarter)",texto)
    if m_quarter:
        tipo_raw = m_quarter.group(1)
        tipo = "last" if "ultim" in tipo_raw or tipo_raw == "last" else "first"
        if not year:
            year = max(available_years) if tipo == "last" else min(available_years)
        start_month, end_month = (1, 3) if tipo == "first" else (10, 12)
        start_date = f"{year}-{start_month:02d}-01"
        end_day = _last_day_of_month(year, end_month)
        end_date = f"{year}-{end_month:02d}-{end_day}"
        return f"fecha_compra BETWEEN '{start_date}' AND '{end_date}'"

    # Meses relativos
    m_months = re.search(
        r"(?:los|las|the)?\s*"
        r"(ultim(?:o|a|os|as)?|primer(?:o|a|os|as)?|last|first)\s+"
        r"(\d{1,2})\s*"
        r"(mes|meses|month|months)"
        r"(?:\s+de\s+(\d{4}))?",
        texto
        )
    
    if m_months:
        tipo_raw = m_months.group(1)
        n_months = int(m_months.group(2))
        n_months = min(max(n_months, 1), 12)

        tipo = "last" if ("ultim" in tipo_raw or tipo_raw == "last") else "first"

        year_in_match = m_months.group(4)
        if year_in_match:
            year = int(year_in_match)
            
        elif not year:
            year = max(available_years) if tipo == "last" else min(available_years)

        if tipo == "first":
            start_month, end_month = 1, n_months
        else:
            start_month, end_month = 12 - n_months + 1, 12
            
        start_date = f"{year}-{start_month:02d}-01"
        end_day = _last_day_of_month(year, end_month)
        end_date = f"{year}-{end_month:02d}-{end_day}"
        return f"fecha_compra BETWEEN '{start_date}' AND '{end_date}'"

    # Año completo
    m_year = re.search(r"(ultim(?:o|a|os|as)?|primer(?:o|a|os|as)?|last|first)\s+(ano|año|year)", texto)
    if m_year:
        tipo_raw = m_year.group(1)
        tipo = "last" if "ultim" in tipo_raw or tipo_raw == "last" else "first"
        if not year:
            year = max(available_years) if tipo == "last" else min(available_years)
        return f"fecha_compra BETWEEN '{year}-01-01' AND '{year}-12-31'"

    return None

In [851]:
COMPARATIVE_WORDS = { "es": { "mas", "menos", "mayor", "menores", "mayores", "vendido", "vendidos", "vendida", "vendidas", "popular", "populares" }, 
                     "en": { "most", "least", "highest", "lowest", "sold", "popular" } } 

In [853]:
def _is_comparative(val: str, idioma: str) -> bool: 
    tokens = val.split()
    return any(t in COMPARATIVE_WORDS[idioma] for t in tokens)

## Ranking

### Filtro ranking

In [775]:
def detectar_ranking(texto: str):
    texto_low = strip_accents(texto.lower())

    # Top explícito
    m_top = re.search(r"\btop\s+(\d+)", texto_low)
    if m_top:
        return True, "DESC", int(m_top.group(1))

    # Singular implícito → "más X" / "menos X"
    if re.search(r"\b(el|la)\s+\w+(?:\s+\w+)*\s+mas\b", texto_low):
        return True, "DESC", 1
    if re.search(r"\b(el|la)\s+\w+(?:\s+\w+)*\s+menos\b", texto_low):
        return True, "ASC", 1

    # Plural implícito → top por defecto
    if re.search(r"\b(los|las)\s+\w+(?:\s+\w+)*\s+mas\b", texto_low):
        return True, "DESC", 5
    if re.search(r"\b(los|las)\s+\w+(?:\s+\w+)*\s+mas\b", texto_low):
        return True, "ASC", 5

    # Inglés
    if re.search(r"\bthe\s+\w+(?:\s+\w+)*\s+most\b", texto_low):
        return True, "DESC", 1
    if re.search(r"\bthe\s+\w+(?:\s+\w+)*\s+least\b", texto_low):
        return True, "ASC", 1

    return False, None, None


### Ranking implicito

In [832]:
def detectar_ranking_implicito(texto, idioma):
    texto = strip_accents(texto.lower())

    if re.search(r"\b(mayor(?:es)?|menor(?:es)?|mas|menos|over|under)\s+de?\s*\d{1,3}\b", texto) and \
       re.search(r"\b(edad|anos|año|years?|cliente|clientes)\b", texto):
        return False

    patrones = [
        r"mas\s+\w+",         
        r"menos\s+\w+",        
        r"mayor(es)?",         
        r"menor(es)?",
        r"highest|lowest|most|least"
    ]

    return any(re.search(p, texto) for p in patrones)

### Ranking

In [778]:
def detectar_ranking_semantico(texto: str, idioma: str):
    t = strip_accents(texto.lower())

    for metric, phrases in RANKING_WORDS[idioma].items():
        for p in phrases:
            if p in t:
                return "sum", metric

    return None

## Dimesión del ranking

In [779]:
def detectar_dimension_ranking(tokens, idioma):
    tokens_norm = [strip_accents(t.lower()) for t in tokens]
    syn_norm = {strip_accents(k.lower()): v for k, v in SYN_TO_COL[idioma].items()}

    for tok in tokens_norm:
        if tok in syn_norm:
            col = syn_norm[tok]
            if col in {"pais", "producto", "id_cliente", "ciudad", "categoria_producto"}:
                return col
    return None

## Conteo de compras

In [780]:
def detectar_conteo_compras(tokens, idioma):
    palabras = {"es": {"compra", "compras", "pedido", "pedidos"},
                "en": {"purchase", "purchases", "orders"}}

    return any(t in palabras[idioma] for t in tokens)

## Filtro agrupaciones

In [830]:
def detectar_agregacion(tokens, idioma):
    # default: None (si no pide nada, se puede devolver *)
    text = " ".join(tokens)
    if re.search(r"\b(mayor(?:es)?|menor(?:es)?|mas|menos|over|under)\s+de?\s*\d{1,3}\b", text):
        if re.search(r"\b(edad|ano|anos|years?|cliente|clientes)\b", text):
            return None

    for agg, words in AGG_WORDS[idioma].items():
        for w in words:
            w_norm = strip_accents(w.lower())
            # si es una frase, búscala en el texto; si es una palabra, también vale con tokens
            if " " in w_norm:
                if w_norm in text:
                    return agg
            else:
                if w_norm in tokens:
                    return agg
    return None

## Detección de where

In [782]:
def detectar_metricas(tokens, idioma):
    # Busca la primera métrica "razonable"
    # Si menciona ventas/importe -> importe_total; unidades -> cantidad; etc.
    for tok in tokens:
        if tok in SYN_TO_COL[idioma]:
            col = SYN_TO_COL[idioma][tok]
            if col in {"importe_total", "cantidad", "precio_unitario", "coste_envio", "coste_fabricacion"}:
                return col
    # fallback: si menciona ventas/total en cualquier parte del texto, asumimos importe_total
    texto = " ".join(tokens)
    if re.search(r"\b(venta|ventas|revenue|ingresos|total|totales)\b", texto.lower()):
        return "importe_total"
    return None

## Detección de group

In [1018]:
def detectar_groupbys(tokens, idioma, filtros=None):
    group_cols = []

    # Tiempo
    time_map = {
        "quarter": ("date_trunc('quarter', fecha_compra)", "trimestre"),
        "month": ("date_trunc('month', fecha_compra)", "mes"),
        "year": ("date_trunc('year', fecha_compra)", "anio"),
    }

    tokens_norm = [strip_accents(t.lower()) for t in tokens]
    
    # Si hay contexto de edad, NO interpretar "año" como agrupación temporal
    text_norm = strip_accents(" ".join(tokens).lower())
    age_context = ("edad" in tokens_norm) or bool(
        re.search(r"\b(entre|between)\s+\d{1,3}\s+(y|and)\s+\d{1,3}\s*(anos|año|years?)\b", text_norm
                  ))

    # Agrupaciones temporales
    relative_month_span = bool(re.search(
        r"(ultim(?:o|a|os|as)?|primer(?:o|a|os|as)?|last|first)\s+\d{1,2}\s+(meses?|months?)",
        text_norm
    ))
    
    if not age_context and not relative_month_span:
        for granularity, (expr, alias) in time_map.items():
            wanted = [strip_accents(w) for w in TIME_GROUP_WORDS[idioma][granularity]]
            if any(t in wanted for t in tokens_norm):
                group_cols.append(expr)
                break

    # Normalizar SYN_TO_COL
    syn_norm = {strip_accents(k.lower()): v for k, v in SYN_TO_COL[idioma].items()}

    # Detectar dimensiones mencionadas usando trigger ("por"/"by")
    trigger_words = {"es": {"por"}, "en": "by"}
    trigger = trigger_words[idioma]

    for i, tok in enumerate(tokens_norm):
        if tok in trigger and i + 1 < len(tokens_norm):
            next_tok = tokens_norm[i + 1]
            if next_tok in syn_norm:
                col = syn_norm[next_tok]
                if col in COLS and col not in group_cols:
                    group_cols.append(col)

    return group_cols

### Función de métrica

In [799]:
def agg_func(metric):
    if metric == "id_transaccion":
        return "COUNT(DISTINCT id_transaccion)"
    if metric == "id_cliente":
        return "COUNT(DISTINCT id_cliente)"
    if metric == "cantidad":
        return "SUM(cantidad)"
    if metric == "importe_total":
        return "SUM(importe_total)"
    return None

In [1150]:
def agg_expr_sql(agg: str, metric: str) -> str:
    if agg == "count":
        return "COUNT(DISTINCT id_transaccion)" if metric == "id_transaccion" else f"COUNT(DISTINCT {metric})"
    if agg == "sum":
        return f"SUM({metric})"
    if agg == "avg":
        return f"AVG({metric})"
    if agg == "max":
        return f"MAX({metric})"
    if agg == "min":
        return f"MIN({metric})"
    if agg == "std":
        return f"STDDEV_POP({metric})"
    if agg == "var":
        return f"VAR_POP({metric})"
    if agg == "median":
        return f"PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY {metric})"
    if agg == "mode":
        return f"MODE() WITHIN GROUP (ORDER BY {metric})"
    return None

## Detección de having

In [841]:
def detectar_having(texto: str):
    texto = strip_accents(texto.lower())

    patrones = [
        # Español (variantes frecuentes)
        (r"\b(mas|mayor(?:es)?|superior(?:es)?)\s+(?:a|de|que)\s+(\d+)\b", ">"),
        (r"\b(menos|menor(?:es)?|inferior(?:es)?)\s+(?:a|de|que)\s+(\d+)\b", "<"),
        (r"\b(?:por\s+encima\s+de|por\s+debajo\s+de)\s+(\d+)\b", None),  # se resuelve abajo

        # Inglés
        (r"\b(greater\s+than|more\s+than|above|over)\s+(\d+)\b", ">"),
        (r"\b(less\s+than|below|under)\s+(\d+)\b", "<"),

        # Operadores explícitos
        (r"\b(>=|<=|=|>|<)\s*(\d+)\b", None),
    ]

    for pat, op in patrones:
        m = re.search(pat, texto)
        if not m:
            continue

        # Caso "por encima/debajo de N"
        if op is None and "por encima de" in m.group(0):
            return ">", int(m.group(1))
        if op is None and "por debajo de" in m.group(0):
            return "<", int(m.group(1))

        valor = int(m.group(2))
        operador = op or m.group(1)
        return operador, valor

    return None

In [865]:
def detectar_having_rango(texto: str):
    t = strip_accents(texto.lower())
    m = re.search(r"\bentre\s+(\d+)\s+y\s+(\d+)\b", t) or re.search(r"\bbetween\s+(\d+)\s+and\s+(\d+)\b", t)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        lo, hi = sorted([a, b])
        return lo, hi
    return None

## Detección de filtro

In [ ]:
def detectar_filtros(doc, tokens, idioma):
    has_relative = False
    has_month_range = False
    has_single_month = False
    
    where = []
    text = strip_accents(doc.text.lower())  # todo en minúscula y sin tildes

    # Fechas relativas (prioridad absoluta)
    relative = _relative_time_condition(doc, idioma)
    if relative:
        where.append(relative)
        has_relative = True

    # Rango de meses
    month_ranges = _find_month_ranges(text, idioma)
    if month_ranges:
        for m_start, m_end, year in month_ranges:
            if not year:
                years_in_text = _find_years(text)
                year = int(years_in_text[0]) if years_in_text else 2024
            start_num = MONTHS[idioma][strip_accents(m_start.lower())]
            end_num = MONTHS[idioma][strip_accents(m_end.lower())]
            where.append(_month_range_condition_pg(start_num, end_num, year))
        has_month_range = True
    
    # Mes en concreto
    single_month = _find_single_month(text, idioma)
    if single_month and not has_month_range:
        month, year = single_month
        if not year:
            years_in_text = _find_years(text)
            year = int(years_in_text[0]) if years_in_text else 2024
        where.append(_single_month_condition_pg(month, year))
        has_single_month = True

    # Año(s) solo si no hay fecha relativa
    if not has_relative and not has_month_range and not has_single_month:
        years = _find_years(text)
        if years:
            where.append(_year_range_condition_pg(years))

    # País / ciudad
    for ent in doc.ents:
        ent_text_norm = strip_accents(ent.text.lower())
        if ent.label_ in {"LOC", "GPE"}: # Aquí si pones el país en minus puedes fallar
            span_start = max(ent.start - 2, 0)
            span_end = min(ent.end + 2, len(doc))
            window = " ".join([strip_accents(t.lemma_.lower()) for t in doc[span_start:span_end]])
            if ("ciudad" in window) or ("city" in window):
                where.append(f"ciudad = '{ent_text_norm}'")
            else:
                pais_real = PAIS_MAP.get(ent_text_norm, ent.text)
                where.append(f"pais = '{pais_real}'")

    # Producto
    m_prod = re.search(r"(producto)\s+([a-z0-9_\-áéíóúñ ]{2,})", text)
    if m_prod:
        val = m_prod.group(2).strip()
        val = re.split(r"\b(en|por|de|del|la|el|and|by|of)\b", val)[0].strip()
        val = strip_accents(val.lower())

        if not _is_comparative(val, idioma):
            cat_real = CATEGORIA_MAP.get(val)
            if cat_real:
                where.append(f"categoria_producto = '{cat_real}'")
            else:
                where.append("producto LIKE '%" + val.replace("'", "''") + "%'")
    # Categoría
    m_cat = re.search(r"(categor[ií]a)\s+([a-z0-9_\-áéíóúñ ]{2,})", text)
    if m_cat:
        val = m_cat.group(2).strip()
        val = re.split(r"\b(en|por|de|del|la|el|and|by|of)\b", val)[0].strip()
        val = strip_accents(val.lower())

        if not _is_comparative(val, idioma):
            cat_real = CATEGORIA_MAP.get(val)
            if cat_real:
                where.append(f"categoria_producto = '{cat_real}'")
            else:
                where.append("categoria producto LIKE '%" + val.replace("'", "''") + "%'")

    # Género
    if re.search(r"\b(masculino|macho?s|varon?es|hombre|hombres|male|m)\b", text):
        where.append("genero IN ('M')")
    if re.search(r"\b(femenino|hembra?s|mujer|mujeres|female|f)\b", text):
        where.append("genero IN ('F')")

    # Entre años
    m_between_age = re.search(r"\b(entre|between)\s+(\d{1,3})\s+(y|and)\s+(\d{1,3})\s*(anos|año|anos|years?)\b", text)
    if m_between_age:
        a = int(m_between_age.group(2))
        b = int(m_between_age.group(4))
        lo, hi = sorted([a, b])
        where.append(f"edad BETWEEN {lo} AND {hi}")

    # Edad
    age_context = bool(re.search(r"\b(edad|anos|año|years?)\b", text)) or bool(re.search(r"\bclientes?\b", text))
    m_gt = re.search(r"\b(mayores de|mas de|over|older than)\s+(\d{1,3})\b", text)
    if m_gt and age_context:
        where.append(f"edad > {int(m_gt.group(2))}")
    m_lt = re.search(r"\b(menores de|menos de|under|younger than)\s+(\d{1,3})\b", text)
    if m_lt and age_context:
        where.append(f"edad < {int(m_lt.group(2))}")

    # Dedup
    where_out = []
    seen = set()
    for w in where:
        if w not in seen:
            where_out.append(w)
            seen.add(w)

    return where_out


## Generador de SQL

In [1153]:
def generar_sql(texto: str):
    doc, idioma = detectar_idioma(texto)
    tokens_norm = _normalize_tokens(doc)
        
    # Detectar GROUP BY, agregación y métricas
    where = detectar_filtros(doc, tokens_norm, idioma)
    group_by = detectar_groupbys(tokens_norm, idioma, filtros=where)

    # Modo listado de clientes (sin agregación)
    wants_clients = ("cliente" in tokens_norm or "clientes" in tokens_norm)
    has_age_filter = any(w.startswith("edad ") for w in where)
    t = strip_accents(texto.lower())

    forced_metric = False
    if re.search(r"\b(numero|número|cuantos?|cuantas?)\b", t) and re.search(r"\bcliente(s)?\b", t):
        agg = "count"
        metric = "id_cliente"
        forced_metric = True

    if wants_clients and group_by:
        agg = "count"
        metric = "id_cliente"

    if not forced_metric:
        agg = detectar_agregacion(tokens_norm, idioma)
        metric = detectar_metricas(tokens_norm, idioma)

    if wants_clients and has_age_filter and (not group_by) and not detectar_agregacion(tokens_norm, idioma):
        agg = None
        metric = None
        group_by = []
    
    if wants_clients and re.search(r"\b(compras?|pedidos?|transacciones?)\b", t):
        agg = "count"
        metric = "id_transaccion"
        if not group_by:
            group_by = ["id_cliente"]

    if re.search(r"\bmes(es)?\b", t) and re.search(r"\b(mas|más)\b", t) and re.search(r"\b(ventas?|importe|ingresos?)\b", t):
        group_by = ["date_trunc('month', fecha_compra)"]
        agg = "sum"
        metric = "importe_total"
        has_ranking = True
        ranking_order = "DESC"
        limit = 12 
    
    if wants_clients and re.search(r"\b(mujer|mujeres|female|femenino)\b", t):
        agg = "count"
        metric = "id_cliente"
        group_by = []
    
    if re.search(r"\bmes\b", t) and re.search(r"\b(menos)\b", t) and re.search(r"\b(2024)\b", t):
        group_by = ["date_trunc('month', fecha_compra)"]
        agg = "sum"
        metric = "importe_total"
        has_ranking = True
        ranking_order = "ASC"
        limit = 1

    has_ranking, ranking_order, limit= detectar_ranking(texto)
    ranking_impl = detectar_ranking_semantico(texto, idioma)
    dimension = detectar_dimension_ranking(tokens_norm, idioma)
    has_ranking_implicito = detectar_ranking_implicito(texto, idioma)

    if ranking_impl:
        agg, metric = ranking_impl
    else:
        if not forced_metric:
            agg = detectar_agregacion(tokens_norm, idioma)
            metric = detectar_metricas(tokens_norm, idioma)
    
    if (not forced_metric) and detectar_conteo_compras(tokens_norm, idioma):
        agg = "count"
        metric = "id_transaccion"

    if (has_ranking or has_ranking_implicito) and dimension:
        group_by = [dimension]
        agg = agg or "sum"
        metric = metric or "cantidad"

    if has_ranking and not group_by:
        if "producto" in tokens_norm or "productos" in tokens_norm:
            group_by = ["producto"]
        elif "cliente" in tokens_norm or "clientes" in tokens_norm:
            group_by = ["id_cliente"]
        elif "pais" in tokens_norm or "paises" in tokens_norm:
            group_by = ["pais"]

    # Ajustes razonables por defecto
    if agg == "count" and not metric:
        metric = "id_transaccion"
    if agg in {"avg", "sum", "max", "min"} and metric is None:
        metric = "importe_total"
    if not metric and group_by:
        metric = "importe_total"
        agg = "sum"
    if agg is None and metric is not None:
        if metric in {"importe_total", "cantidad"}:
            agg = "sum"
    
    having_range = detectar_having_rango(texto)
    having_cond = detectar_having(texto)
    having = []

    has_age_filter = any(w.startswith("edad ") for w in where)  # ya lo tienes arriba, reutilízalo si quieres
    if has_age_filter:
        having_cond = None
        having_range = None

    if having_cond:
        has_ranking = False
        has_ranking_implicito = False
        ranking_order = None
        limit = None

    if having_range and re.search(r"\b(compras?|pedidos?|transacciones?)\b", strip_accents(texto.lower())):
        agg = "count"
        metric = "id_transaccion"
        lo, hi = having_range
        having.append(f"{agg_expr_sql(agg, metric)} BETWEEN {lo} AND {hi}")
        if wants_clients and (having_cond or having_range) and not group_by:
            group_by = ["id_cliente"]

    if detectar_conteo_compras(tokens_norm, idioma):
        agg = "count"
        metric = "id_transaccion"
    
    if (not forced_metric) and agg == "count" and metric != "id_cliente":
        metric = "id_transaccion"
    
    if agg is not None:
        agg = agg or "sum"
        metric = metric or "importe_total"

    if having_cond:
        operador, valor = having_cond
        expr = agg_expr_sql(agg, metric)
        if expr:
            having.append(f"{expr} {operador} {valor}")
    
    # SELECT
    alias_metric = None
    select_parts = []
    if group_by:
        select_parts.extend(group_by)
    if agg == "var" and metric not in {"importe_total", "cantidad", "precio_unitario", "coste_envio", "coste_fabricacion"}:
        metric = "importe_total"
    if agg == "avg":
        alias_metric = f"promedio_{metric}"
        select_parts.append(f"AVG({metric}) AS {alias_metric}")
    elif agg == "sum":
        alias_metric = f"total_{metric}"
        select_parts.append(f"{agg_func(metric)} AS {alias_metric}")
    elif agg == "max":
        alias_metric = f"max_{metric}"
        select_parts.append(f"MAX({metric}) AS {alias_metric}")
    elif agg == "min":
        alias_metric = f"min_{metric}"
        select_parts.append(f"MIN({metric}) AS {alias_metric}")
    elif agg == "median":
        alias_metric = f"mediana_{metric}"
        select_parts.append(f"PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY {metric}) AS {alias_metric}")
    elif agg == "mode":
        alias_metric = f"moda_{metric}"
        select_parts.append(f"MODE() WITHIN GROUP (ORDER BY {metric}) AS {alias_metric}")
    elif agg == "std":
        alias_metric = f"std_{metric}"
        select_parts.append(f"STDDEV_POP({metric}) AS {alias_metric}")
    elif agg == "var":
        alias_metric = f"var_{metric}"
        select_parts.append(f"VAR_POP({metric}) AS {alias_metric}")
    elif agg == "count":
        # Si pide conteo, contamos transacciones por defecto
        alias_metric = "conteo_clientes" if metric == "id_cliente" else "conteo_transacciones"
        select_parts.append(f"{agg_func(metric)} AS {alias_metric}")
    else:
        if not group_by:
            if wants_clients:
                select_parts.extend(["id_cliente", "nombre", "apellidos", "edad", "pais", "ciudad", "email"])
            else:
                select_parts.extend(["id_transaccion", "fecha_compra", "importe_total", "cantidad"])
    

    sql = "SELECT " + ", ".join(select_parts) + f" FROM {TABLE_NAME}"

    if where:
        sql += " WHERE " + " AND ".join(where)

    if group_by:
        sql += " GROUP BY " + ", ".join(group_by)
    
    if having:
        sql += " HAVING " + " AND ".join(having)

    if (has_ranking or has_ranking_implicito) and dimension:
        sql += f" ORDER BY {alias_metric} {ranking_order or 'DESC'}"
        if not limit and has_ranking_implicito:
            if re.search(r"\b(productos|clientes|paises|categorias|ciudades)\b", strip_accents(texto.lower())):
                limit = 5
                
        sql += f" LIMIT {limit or 1}"
            
    sql += ";"
    return sql

## Pruebas

LEFT JOIN --> De transacciones a cliente
PROS:
- clientes.id_cliente es PK
- transaccion.id_transacciones es PK
- transaccion.id_cliente es FK
- No hay duplicados reales con LEFT JOIN.

FULL OTER --> Genera filas fantasmas
CONTRA:
- COUNT(id_transaccion)
- SUM(importe_total)
- rankings
- métricas BI

### 1️⃣ Fechas relativas (mes / trimestre / año) ✅✅

In [887]:
print(generar_sql("¿Cuántas transacciones hubo en el primer trimestre de 2024?"))

SELECT date_trunc('quarter', fecha_compra), SUM(cantidad) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-03-31' GROUP BY date_trunc('quarter', fecha_compra);


In [1013]:
print(generar_sql("Promedio de ventas entre febrero y abril de 2024"))

SELECT AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-02-01' AND '2024-04-30';


In [1019]:
print(generar_sql("Promedio de ventas de los primeros 2 meses de 2024"))

SELECT date_trunc('month', fecha_compra), AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-02-29' GROUP BY date_trunc('month', fecha_compra);


In [1020]:
print(generar_sql("Total de ventas de los últimos 6 meses de 2023"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-07-01' AND '2023-12-31' GROUP BY date_trunc('month', fecha_compra);


In [891]:
print(generar_sql("Número de transacciones en el último año"))

SELECT date_trunc('year', fecha_compra), SUM(cantidad) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY date_trunc('year', fecha_compra);


In [892]:
print(generar_sql("Ventas del 2023"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31';


In [893]:
print(generar_sql("Las ventas en el 2024"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31';


### 2️⃣ Fechas absolutas + group by temporal ✅

In [894]:
print(generar_sql("Ventas totales en enero de 2024"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-01-31' GROUP BY date_trunc('month', fecha_compra);


In [895]:
print(generar_sql("Número de transacciones en marzo de 2023"))

SELECT date_trunc('month', fecha_compra), SUM(cantidad) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-03-01' AND '2023-03-31' GROUP BY date_trunc('month', fecha_compra);


In [1021]:
print(generar_sql("Ventas totales entre junio y septiembre de 2023 por mes"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-06-01' AND '2023-09-30' GROUP BY date_trunc('month', fecha_compra);


In [1104]:
print(generar_sql("Número de transacciones en 2024 por trimestre"))

SELECT date_trunc('quarter', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY date_trunc('quarter', fecha_compra);


In [1105]:
print(generar_sql("Promedio de ventas en 2023 por mes"))

SELECT date_trunc('month', fecha_compra), AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' GROUP BY date_trunc('month', fecha_compra);


### 3️⃣ País / ciudad (mapeo + NER) ✅✅

In [900]:
print(generar_sql("Cuántas transacciones hubo en España en 2024"))

SELECT SUM(cantidad) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND pais = 'España';


In [901]:
print(generar_sql("Ventas totales en México por mes en 2023"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'México' GROUP BY date_trunc('month', fecha_compra);


In [903]:
print(generar_sql("Promedio de ventas en Francia en el primero trimestre de 2024"))

SELECT date_trunc('quarter', fecha_compra), AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-03-31' AND pais = 'Francia' GROUP BY date_trunc('quarter', fecha_compra);


In [1023]:
print(generar_sql("Ventas totales en Argentina en los últimos 3 meses de 2023"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-10-01' AND '2023-12-31' AND pais = 'Argentina' GROUP BY date_trunc('month', fecha_compra);


In [1099]:
print(generar_sql("Número de transacciones en Portugal por trimestre"))

SELECT date_trunc('quarter', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE pais = 'Portugal' GROUP BY date_trunc('quarter', fecha_compra);


In [906]:
print(generar_sql("Ventas por país en 2024"))

SELECT pais, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY pais;


### 4️⃣ Métricas distintas (sum, avg, max, min, std, median) ✅✅

In [907]:
print(generar_sql("Promedio de ventas en 2024"))

SELECT AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31';


In [908]:
print(generar_sql("Ventas máximas en España en 2023"))

SELECT MAX(importe_total) AS max_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'España';


In [909]:
print(generar_sql("Mínimo ventas en México en 2024"))

SELECT MIN(importe_total) AS min_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND pais = 'México';


In [910]:
print(generar_sql("Desviación estándar de ventas en 2023"))

SELECT STDDEV_POP(importe_total) AS std_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31';


In [911]:
print(generar_sql("Mediana de ventas en el último trimestre de 2024"))

SELECT date_trunc('quarter', fecha_compra), PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY importe_total) AS mediana_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-10-01' AND '2024-12-31' GROUP BY date_trunc('quarter', fecha_compra);


In [915]:
print(generar_sql("Moda de las ventas en 2023"))

SELECT MODE() WITHIN GROUP (ORDER BY importe_total) AS moda_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31';


In [916]:
print(generar_sql("El producto más vendido en 2023"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 1;


In [917]:
print(generar_sql("Los productos más vendidos del 2024"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 5;


In [918]:
print(generar_sql("Productos más vendidos del último trimestre de 2023"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-10-01' AND '2023-12-31' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 1;


In [919]:
print(generar_sql("Productos más vendidos del último trimestre"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-10-01' AND '2024-12-31' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 1;


In [1154]:
print(generar_sql("Varianza de ventas en 2024"))

SELECT VAR_POP(importe_total) AS var_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31';


In [1155]:
print(generar_sql("Varianza del importe total por mes en 2024"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY date_trunc('month', fecha_compra);


### 5️⃣ Count (con y sin group by) ✅

In [920]:
print(generar_sql("Cuántas transacciones hubo en 2023"))

SELECT SUM(cantidad) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31';


In [1106]:
print(generar_sql("Número de transacciones por mes en 2024"))

SELECT date_trunc('month', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY date_trunc('month', fecha_compra);


In [1107]:
print(generar_sql("Cuántas transacciones hubo en España por trimestre en 2023"))

SELECT date_trunc('quarter', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'España' GROUP BY date_trunc('quarter', fecha_compra);


In [923]:
print(generar_sql("Número de pedidos en México en el primer trimestre de 2024"))

SELECT date_trunc('quarter', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-03-31' AND pais = 'México' GROUP BY date_trunc('quarter', fecha_compra);


In [924]:
print(generar_sql("Cuántas transacciones por país en 2024"))

SELECT pais, SUM(cantidad) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY pais;


### 6️⃣ Ranking / Top N ✅✅

In [925]:
print(generar_sql("Top 5 productos más vendidos en 2024"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 5;


In [926]:
print(generar_sql("Top 3 países con mayores ventas en 2023"))

SELECT pais, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' GROUP BY pais ORDER BY total_importe_total DESC LIMIT 3;


In [927]:
print(generar_sql("Top 10 productos por ingresos en 2024"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 10;


In [928]:
print(generar_sql("Productos más vendidos del último trimestre de 2023"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-10-01' AND '2023-12-31' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 1;


In [531]:
print(generar_sql("Top 5 categorías con mayor facturación en 2024"))

SELECT categoria_producto, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY categoria_producto ORDER BY total_importe_total DESC LIMIT 5;


In [929]:
print(generar_sql("Top 3 países por volumen de ventas en 2023"))

SELECT pais, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' GROUP BY pais ORDER BY total_cantidad DESC LIMIT 3;


### 7️⃣ Ranking + filtros temporales ✅✅

In [930]:
print(generar_sql("Top 5 productos más vendidos en España en 2024"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND pais = 'España' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 5;


In [931]:
print(generar_sql("Top 3 países con más ingresos en el primer trimestre de 2023"))

SELECT pais, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-03-31' GROUP BY pais ORDER BY total_importe_total DESC LIMIT 3;


In [1111]:
print(generar_sql("Top 10 productos más vendidos en los últimos 3 meses de 2024"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-10-01' AND '2024-12-31' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 10;


In [1110]:
print(generar_sql("productos más vendidos en México en 2023"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'México' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 5;


In [1109]:
print(generar_sql("Top 5 ciudades con más ventas de los últimos 3 meses de 2024"))

SELECT ciudad, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-10-01' AND '2024-12-31' GROUP BY ciudad ORDER BY total_importe_total DESC LIMIT 5;


### 8️⃣ Filtros demográficos ✅

In [1127]:
print(generar_sql("Ventas totales a mujeres en 2023"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND genero IN ('F');


In [1128]:
print(generar_sql("Promedio de ventas de hombres en España en 2024"))

SELECT AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND genero IN ('M');


In [1129]:
print(generar_sql("Número de transacciones de clientes mayores de 40 en México"))

SELECT id_cliente, COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE pais = 'México' AND edad > 40 GROUP BY id_cliente;


In [1130]:
## TOP
print(generar_sql("Ventas totales por género en 2024"))

SELECT genero, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY genero;


### 9️⃣ Producto / categoría ✅✅

In [1148]:
print(generar_sql("Ventas totales del producto movil en 2024"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND categoria_producto = 'Móviles';


In [1147]:
print(generar_sql("Número de transacciones de la categoría reloj en 2023"))

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND categoria_producto = 'Relojes inteligentes';


In [1146]:
## TOP
print(generar_sql("Promedio de ventas por categoría en 2024"))

SELECT categoria_producto, AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND categoria_producto LIKE '%%' GROUP BY categoria_producto;


In [948]:
print(generar_sql("Top 5 productos más vendidos de la categoría accesorios"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE categoria_producto = 'Accesorios' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 5;


In [1145]:
print(generar_sql("Ventas totales por categoría en España en 2023"))

SELECT categoria_producto, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'España' AND categoria_producto LIKE '%%' GROUP BY categoria_producto;


### 🔟 Inglés (para validar bilingüe completo) ✅✅

In [950]:
print(generar_sql("How many transactions were there in 2024?"))

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31';


In [951]:
print(generar_sql("Total sales by country in 2023"))

SELECT pais, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' GROUP BY pais;


In [1116]:
print(generar_sql("Average sales per month in 2024"))

SELECT date_trunc('month', fecha_compra), AVG(importe_total) AS promedio_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY date_trunc('month', fecha_compra);


In [953]:
print(generar_sql("Top 5 best selling products in 2024"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 5;


In [954]:
print(generar_sql("Number of transactions in Spain in the first quarter of 2023"))

SELECT date_trunc('quarter', fecha_compra), SUM(cantidad) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-03-31' AND pais = 'España' GROUP BY date_trunc('quarter', fecha_compra);


In [1054]:
print(generar_sql("Total revenue in the last 3 months of 2024"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-10-01' AND '2024-12-31';


In [1144]:
print(generar_sql("How many transactions in the UK in 2023?"))

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'Reino Unido';


### 1️⃣1️⃣ Edge cases interesantes ✅

In [1143]:
print(generar_sql("Cuántas ventas hubo"))

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente;


In [964]:
print(generar_sql("Dame las ventas por mes"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente GROUP BY date_trunc('month', fecha_compra);


In [1064]:
print(generar_sql("Cuántos clientes tuve")) 

SELECT COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente;


In [967]:
print(generar_sql("Ventas por país"))

SELECT pais, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente GROUP BY pais;


In [968]:
print(generar_sql("Ventas en España"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE pais = 'España';


In [1103]:
print(generar_sql("¿Cuántas transacciones hubo en España en el año 2023?"))

SELECT date_trunc('year', fecha_compra), COUNT(DISTINCT id_transaccion) AS conteo_transacciones FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' AND pais = 'España' GROUP BY date_trunc('year', fecha_compra);


In [1097]:
print(generar_sql("Las ventas en 2024 en España"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' AND pais = 'España';


# RAQUEL

In [1067]:
print(generar_sql("productos mas vendidos en agosto 2024"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-08-01' AND '2024-08-31' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 5;


In [971]:
print(generar_sql("producto mas vendido del 2023"))

SELECT producto, SUM(cantidad) AS total_cantidad FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2023-01-01' AND '2023-12-31' GROUP BY producto ORDER BY total_cantidad DESC LIMIT 1;


In [607]:
print(generar_sql("Dame las ventas totales de febrero"))

SELECT date_trunc('month', fecha_compra), SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-02-01' AND '2024-02-29' GROUP BY date_trunc('month', fecha_compra);


In [823]:
print(generar_sql("clientes entre 18 y 30 años"))

SELECT id_cliente, nombre, apellidos, edad, pais, ciudad, email FROM merge_transaccion_cliente WHERE edad BETWEEN 18 AND 30;


In [822]:
print(generar_sql("clientes entre 18 y 30 años en España"))

SELECT id_cliente, nombre, apellidos, edad, pais, ciudad, email FROM merge_transaccion_cliente WHERE pais = 'España' AND edad BETWEEN 18 AND 30;


In [1093]:
print(generar_sql("clientes mayores de 40"))

SELECT id_cliente, nombre, apellidos, edad, pais, ciudad, email FROM merge_transaccion_cliente WHERE edad > 40;


#### HAVING

In [1068]:
print(generar_sql("productos con ventas mayores a 100 en 2024"))

SELECT producto, SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente WHERE fecha_compra BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY producto HAVING SUM(importe_total) > 100;


In [1094]:
print(generar_sql("ventas mayores de 100"))

SELECT SUM(importe_total) AS total_importe_total FROM merge_transaccion_cliente HAVING SUM(importe_total) > 100;


---